In [1]:
# ===== Cell 1: Install & imports (Colab) =====
!pip -q install pytorchvideo decord opencv-contrib-python tqdm scikit-learn

import os, glob, random, math, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

import decord
from decord import VideoReader, cpu
import cv2

from pytorchvideo.models.hub import x3d_s

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

Torch: 2.5.1 | CUDA available: True


In [2]:
# ===== Cell 2: Config, paths, seeds =====
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# CHANGE THIS IF NEEDED
# For Colab (after mounting Drive):
# from google.colab import drive
# drive.mount('/content/drive')
ROOT = Path(r"E:\Graduation Project")
# For local Windows, you can use:
# ROOT = Path(r"D:\Graduation Project")

BASE_DIR    = ROOT / "violence_data"
ALL_DIR     = BASE_DIR / "all_videos"
SPLITS_DIR  = BASE_DIR / "splits"
MODELS_DIR  = ROOT / "models"

for p in [ALL_DIR/"Violence", ALL_DIR/"NonViolence", SPLITS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Hyperparameters
SEED          = 1337
EPOCHS        = 20
BATCH_SIZE    = 4
NUM_WORKERS   = 0        # set to 0 on Windows if you hit issues
T             = 13       # number of frames per clip
SIZE          = 160      # spatial size
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
NUM_CLASSES   = 2        # NonViolence, Violence

def set_seed(seed=1337):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True

print("Project root:", ROOT)

Using device: cuda
Project root: E:\Graduation Project


In [3]:
# ===== Cell 3: Create train/val/test splits (tab-separated) =====
# This will re-create splits each time. If you want them fixed, run once then comment this cell out later.

rows = []
for label_name, label_id in [("NonViolence", 0), ("Violence", 1)]:
    pattern = str((ALL_DIR / label_name) / "*.mp4")
    files = sorted(glob.glob(pattern))
    print(f"Found {len(files)} videos in {label_name}")
    for f in files:
        rows.append((f, label_id))

random.shuffle(rows)
N = len(rows)
i1, i2 = int(0.7 * N), int(0.85 * N)
train_rows = rows[:i1]
val_rows   = rows[i1:i2]
test_rows  = rows[i2:]

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for name, part in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    with open(SPLITS_DIR / f"{name}.txt", "w", encoding="utf-8") as f:
        for p, l in part:
            f.write(f"{p}\t{l}\n")  # TAB separator

print("Split sizes -> train:", len(train_rows), "val:", len(val_rows), "test:", len(test_rows))

Found 2865 videos in NonViolence
Found 2865 videos in Violence
Split sizes -> train: 4010 val: 860 test: 860


In [4]:
# ===== Cell 4: ClipDataset + DataLoaders + class weights =====
decord.bridge.set_bridge("native")

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class ClipDataset(Dataset):
    def __init__(self, list_file, clip_len=T, frame_size=SIZE, train=True):
        self.samples = []
        with open(list_file, encoding="utf-8") as f:
            for line in f:
                path, label = line.strip().split("\t")  # TAB split (matches the splits we wrote)
                self.samples.append((path, int(label)))
        self.clip_len = clip_len
        self.frame_size = frame_size
        self.train = train

    def _indices(self, n_frames: int):
        if n_frames <= self.clip_len:
            return np.linspace(0, n_frames - 1, self.clip_len).astype(int)
        if self.train:
            start = np.random.randint(0, n_frames - self.clip_len + 1)
        else:
            start = (n_frames - self.clip_len) // 2
        return np.arange(start, start + self.clip_len)

    def _resize_crop(self, img: np.ndarray):
        h, w = img.shape[:2]
        scale = int(self.frame_size * 1.14)
        short = min(h, w)
        r = scale / float(short)
        img = cv2.resize(img, (int(w * r), int(h * r)))
        h, w = img.shape[:2]

        if self.train:
            y = np.random.randint(0, h - self.frame_size + 1)
            x = np.random.randint(0, w - self.frame_size + 1)
        else:
            y = max((h - self.frame_size) // 2, 0)
            x = max((w - self.frame_size) // 2, 0)

        img = img[y:y + self.frame_size, x:x + self.frame_size]

        if self.train and np.random.rand() < 0.5:
            img = img[:, ::-1]

        return img

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        vr = VideoReader(path, ctx=cpu(0))
        idxs = self._indices(len(vr))
        frames = vr.get_batch(idxs).asnumpy()   # (T, H, W, 3) uint8

        frames = [self._resize_crop(f) for f in frames]
        frames = np.stack(frames).astype(np.float32) / 255.0  # (T, H, W, 3)
        frames = (frames - IMAGENET_MEAN) / IMAGENET_STD
        frames = np.transpose(frames, (3, 0, 1, 2))  # (C, T, H, W)

        return torch.from_numpy(frames), torch.tensor(label, dtype=torch.long)

# Build datasets/loaders
train_ds = ClipDataset(SPLITS_DIR / "train.txt", clip_len=T, frame_size=SIZE, train=True)
val_ds   = ClipDataset(SPLITS_DIR / "val.txt",   clip_len=T, frame_size=SIZE, train=False)
test_ds  = ClipDataset(SPLITS_DIR / "test.txt",  clip_len=T, frame_size=SIZE, train=False)

print("Dataset sizes -> train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# Class weights (to fight imbalance)
labels_train = [lbl for _, lbl in train_ds.samples]
counts = Counter(labels_train)
total = len(labels_train)
class_weights = []
for c in range(NUM_CLASSES):
    # inverse-frequency-ish weighting
    class_weights.append(total / (counts[c] * NUM_CLASSES))

CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class counts:", counts)
print("Class weights:", CLASS_WEIGHTS.cpu().numpy())

Dataset sizes -> train: 4010 val: 860 test: 860
Class counts: Counter({1: 2020, 0: 1990})
Class weights: [1.0075377 0.9925743]


In [5]:
# ===== Cell 5: X3D-S model =====
# Load pretrained X3D-S and replace the final head
model = x3d_s(pretrained=True)

in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"X3D-S total params: {total_params:.2f}M")

X3D-S total params: 2.98M


In [6]:
# ===== Cell 6: Training loop (optimize for Violence recall) =====
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_violence_recall = 0.0
best_state = None

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")
    # ---- Train ----
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for clips, labels in train_loader:
        clips  = clips.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(clips)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss   += loss.item() * labels.size(0)
        preds           = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total   += labels.size(0)

    train_loss = running_loss / max(1, running_total)
    train_acc  = running_correct / max(1, running_total)
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.3f}")

    # ---- Validation ----
    model.eval()
    val_loss_sum = 0.0
    val_total    = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for clips, labels in val_loader:
            clips  = clips.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(clips)
            loss   = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_total    += labels.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1]  # P(violence)
            preds = (probs >= 0.5).long()

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    val_loss = val_loss_sum / max(1, val_total)
    print(f"Val loss: {val_loss:.4f}")

    report_dict = classification_report(
        y_true, y_pred, target_names=["NonViolence", "Violence"],
        digits=4, output_dict=True
    )
    print("Val classification report:")
    print(classification_report(y_true, y_pred, target_names=["NonViolence", "Violence"], digits=4))

    violence_recall = report_dict["Violence"]["recall"]
    print(f"Violence recall this epoch: {violence_recall:.4f}")

    if violence_recall > best_violence_recall:
        best_violence_recall = violence_recall
        best_state = model.state_dict()
        torch.save(best_state, MODELS_DIR / "x3d_s_best.pth")
        print("✅ Saved new best model (by Violence recall).")

    scheduler.step()

print(f"\nBest Violence recall on validation: {best_violence_recall:.4f}")


===== Epoch 1/20 =====
Train loss: 0.5690 | Train acc: 0.732
Val loss: 0.5333
Val classification report:
              precision    recall  f1-score   support

 NonViolence     0.7376    0.8855    0.8048       454
    Violence     0.8349    0.6478    0.7295       406

    accuracy                         0.7733       860
   macro avg     0.7863    0.7666    0.7672       860
weighted avg     0.7836    0.7733    0.7693       860

Violence recall this epoch: 0.6478
✅ Saved new best model (by Violence recall).

===== Epoch 2/20 =====
Train loss: 0.5605 | Train acc: 0.744
Val loss: 0.5877
Val classification report:
              precision    recall  f1-score   support

 NonViolence     0.6480    0.9692    0.7767       454
    Violence     0.9227    0.4113    0.5690       406

    accuracy                         0.7058       860
   macro avg     0.7853    0.6902    0.6728       860
weighted avg     0.7777    0.7058    0.6786       860

Violence recall this epoch: 0.4113

===== Epoch 3/20 =

In [7]:
# ===== Cell 7: Test eval + threshold sweep (choose production threshold) =====
# Reload best model
best_model = x3d_s(pretrained=False)
in_features = best_model.blocks[5].proj.in_features
best_model.blocks[5].proj = nn.Linear(in_features, NUM_CLASSES)
best_model.load_state_dict(torch.load(MODELS_DIR / "x3d_s_best.pth", map_location=DEVICE))
best_model = best_model.to(DEVICE)
best_model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for clips, labels in test_loader:
        clips  = clips.to(DEVICE, non_blocking=True)
        logits = best_model(clips)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

def eval_at_threshold(th):
    preds = (all_probs >= th).astype(int)
    rep = classification_report(
        all_labels, preds, target_names=["NonViolence", "Violence"],
        digits=4, output_dict=True
    )
    return {
        "threshold": float(th),
        "violence_precision": rep["Violence"]["precision"],
        "violence_recall":    rep["Violence"]["recall"],
        "violence_f1":        rep["Violence"]["f1-score"],
        "accuracy":           rep["accuracy"],
    }

thresholds = np.linspace(0.2, 0.6, 9)
results = [eval_at_threshold(t) for t in thresholds]

print("Threshold sweep on TEST set:")
for r in results:
    print(r)

best_by_f1 = max(results, key=lambda x: x["violence_f1"])
print("\n✅ Recommended threshold (max Violence F1):", best_by_f1)

C:\Users\abdal\AppData\Local\Temp\ipykernel_41708\171354200.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model.load_state_dict(torch.load(MODELS_DIR / "x3d_s_bes

Threshold sweep on TEST set:
{'threshold': 0.2, 'violence_precision': 0.5104651162790698, 'violence_recall': 1.0, 'violence_f1': 0.6759045419553503, 'accuracy': 0.5104651162790698}
{'threshold': 0.25, 'violence_precision': 0.5104651162790698, 'violence_recall': 1.0, 'violence_f1': 0.6759045419553503, 'accuracy': 0.5104651162790698}
{'threshold': 0.3, 'violence_precision': 0.6938775510204082, 'violence_recall': 0.929384965831435, 'violence_f1': 0.7945472249269717, 'accuracy': 0.7546511627906977}
{'threshold': 0.35, 'violence_precision': 0.7191413237924866, 'violence_recall': 0.9157175398633257, 'violence_f1': 0.8056112224448898, 'accuracy': 0.7744186046511627}
{'threshold': 0.4, 'violence_precision': 0.7391304347826086, 'violence_recall': 0.8906605922551253, 'violence_f1': 0.8078512396694215, 'accuracy': 0.7837209302325582}
{'threshold': 0.44999999999999996, 'violence_precision': 0.7519685039370079, 'violence_recall': 0.8701594533029613, 'violence_f1': 0.8067581837381204, 'accuracy': 0.

C:\Users\abdal\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\abdal\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\abdal\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [9]:
# ===== Cell 8: Export best model as TorchScript for production =====
from pytorchvideo.layers.swish import Swish as PtvSwish # Import PyTorchVideo's Swish

def replace_pytorchvideo_swish_with_silu(module):
    for name, child in module.named_children():
        if isinstance(child, PtvSwish):
            # Replace PyTorchVideo's Swish with torch.nn.SiLU
            setattr(module, name, nn.SiLU())
        else:
            replace_pytorchvideo_swish_with_silu(child)

example_input = torch.randn(1, 3, T, SIZE, SIZE, device=DEVICE)
best_model.eval()

# Apply the replacement before tracing
replace_pytorchvideo_swish_with_silu(best_model)

traced = torch.jit.trace(best_model, example_input)
export_path = MODELS_DIR / "x3d_s_best_ts.pt"
torch.jit.save(traced, export_path)
print("Saved TorchScript model to:", export_path)

Saved TorchScript model to: E:\Graduation Project\models\x3d_s_best_ts.pt
